# The Data Machine — Guía de la Homepage

**Responsable:** Melanie  
**Archivo:** `pages/homepage.py`  
**Propósito:** Catálogo interactivo de juegos de Steam con análisis de sentimiento VADER y extracción de temas TF-IDF.

## 1. Arquitectura

```
app.py (entrypoint)
  |
  ├─ ¿query param ?game=X?
  │     └─ st.switch_page("pages/analisis.py")
  │
  └─ default
        └─ st.switch_page("pages/homepage.py")
              |
              ├─ auth guard (session_state["authenticated"])
              ├─ import BASE_CSS from src/styles.py
              ├─ load_catalog() → dataset_limpio.csv / demo
              ├─ render sections
              └─ st.components.v1.html() → JS (nebula, música, carrusel)
```

### Páginas del proyecto

| Archivo | Quién | Estado |
|---------|-------|--------|
| `app.py` | — | ✅ Listo — routing + auth debug |
| `pages/homepage.py` | **Melanie** | ✅ Listo |
| `pages/analisis.py` | Compañerx | ⏳ Stub — falta implementar |
| `pages/login.py` | Compañerx | ⏳ Stub — falta implementar |

## 2. Flujo de datos

`load_catalog()` intenta cargar `data/processed/dataset_limpio.csv`.

- **Si existe y tiene columna `name`** → usa datos reales
- **Si falla (archivo no existe o columna faltante)** → demo de 10 juegos hardcodeados

### Columnas esperadas del dataset

| Columna | Tipo | Uso |
|---------|------|-----|
| `name` | string | Nombre del juego |
| `header_image` | string | URL imagen Steam |
| `price` | float | Precio USD (×20 → MXN) |
| `pct_pos_total` | int | % reseñas positivas |
| `tags` | string | Categorías separadas por coma |
| `metacritic_score` | int | Puntuación Metacritic (opcional) |
| `estimated_owners` | string | Rango propietarios (opcional) |

## 3. Secciones de la Homepage

Orden actual de renderizado:

| # | Sección | Cómo se renderiza | Datos que usa |
|---|---------|-------------------|---------------|
| 1 | **Header** | `st.markdown` — topbar con logo gradiente, botón música, usuario | `session_state["username"]` |
| 2 | **Hero** | `st.markdown` — eyebrow con glitch, título 52px, subtítulo, scroll indicator `⟡` | Texto estático |
| 3 | **Stats** | `st.markdown` — 4 columnas con números animados (JS count-up) | `len(df)`, hardcode 21M+ |
| 4 | **¿Qué es?** | `st.markdown` + `st.columns(3)` — título + 3 info cards con letras S/T/D | Texto estático |
| 5 | **Featured Game** | `st.markdown` — juego aleatorio con imagen, overlay, tags, botón | `df.sample(1)` |
| 6 | **Filtros** | `st.columns(4)` — widgets Streamlit: nombre, género, precio, rating | Input usuario |
| 7 | **Categorías** (o Resultados) | `_cat_row_html()` — filas horizontales Netflix-style con scroll y flechas | `df.sample()`, `nlargest()`, filtros |
| 8 | **Tech Stack** | `st.markdown` — badges de tecnologías | Texto estático |
| 9 | **Footer** | `st.markdown` — copyright, créditos | Texto estático |
| — | **Nebula (JS)** | `st.components.v1.html` — canvas fixed en body, 500 partículas, shockwave, estrellas | N/A (JS puro) |
| — | **Música** | Mismo bloque JS — synthwave Web Audio API, toggle con botón | N/A (JS puro) |

## 4. Interactividad

### Nebula (partículas)
- Canvas `position: fixed` en `body` del documento padre
- 500 partículas en órbita, 100 estrellas titilantes
- **Mouse move** → atracción/repulsión de partículas
- **Click global** → onda expansiva (shockwave) en cualquier parte de la página
- Redimensiona con la ventana

### Música synthwave
- Generada con Web Audio API (osciladores, sin archivos externos)
- 8 compases, loop de 16s: kick, snare, arpegiador, pad
- 7ª cuerda aumentada (Cmaj7, Am, Fmaj7, G, Am, C, Dm)
- Botón en header para play/pause

### Filtros de búsqueda
- 4 controles Streamlit en `st.columns(4)`:
  - **Nombre** — `st.text_input`, búsqueda por substring en `name`
  - **Género** — `st.multiselect`, filtro sobre columna `genres` (28 géneros únicos)
  - **Precio MXN** — `st.slider`, rango de precio (USD × 20)
  - **Rating mínimo** — `st.select_slider` con valores 0, 50, 65, 75, 85, 95
- Cuando hay filtros activos → se muestran resultados en una fila horizontal
- Sin filtros → se muestran las categorías predefinidas
- Filtran sobre el dataset completo (~10k juegos)

### Categorías Netflix-style (filas horizontales)
- 10 filas generadas con `_cat_row_html()`:
  1. **⟡ Nuestra selección para ti** — 20 juegos aleatorios
  2. **◆ Top 10 Mejor Valorados** — top `pct_pos_total`
  3. **▸ Recién Llegados** — más recientes por `release_date`
  4. **Gratis** — `price == 0`
  5. **▲ Para aniquilar el aburrimiento** — mayor `avg_playtime`
  6. **★ Mejor relación calidad-precio** — rating ≥ 85 + precio bajo
  7. **⟐ Más Populares** — top `num_reviews_total`
  8. **⬡ Recomendados por crítica** — `metacritic_score ≥ 80`
  9. **⊡ Indies Imperdibles** — género "Indie" + rating alto
  10. **⏷ En Descuento** — `discount > 0`
- Cada fila con:
  - Track horizontal con `overflow-x: auto` y `scroll-behavior: smooth`
  - Flechas ← → (JS: `querySelectorAll('.cat-wrapper')` con scroll por 4 cards)
  - Cards responsivas: 5 visibles en desktop, 4 en tablet, 3 en mobile
  - Sin autoplay ni dots (estilo Netflix)

## 5. Páginas del equipo (Stubs)

### `pages/analisis.py` — Compañerx

**Estado actual:** Stub básico que muestra `selected_game` y hereda `BASE_CSS`.

**Qué debe contener (ideal):**
- Dashboard por juego con:
  - Nube de palabras (TF-IDF)
  - Gráfica de sentimiento (VADER: % positivo/neutral/negativo)
  - Tabla de reseñas recientes
  - Métricas: precio MXN, Metacritic, rating, dueños estimados
- Botón de descarga CSV/PNG
- Botón "Volver al catálogo" que haga `st.switch_page("pages/homepage.py")`

### `pages/login.py` — Compañerx

**Estado actual:** Stub con formulario usuario + contraseña.

**Qué debe contener (ideal):**
- Validación de credenciales contra usuarios hardcodeados o DB
- `st.session_state["authenticated"] = True` al autenticar
- Redirección a `pages/homepage.py`
- Diseño centrado, mismo `BASE_CSS`

## 6. Checklist por responsable

### Melanie — ✅ Homepage

- [x] Hero con glitch y scroll indicator
- [x] Nebula de partículas con shockwave
- [x] Música synthwave toggle
- [x] Stats count-up
- [x] ¿Qué es? con letras decorativas S/T/D
- [x] Juego Destacado
- [x] Carrusel reemplazado por 10 categorías Netflix-style + flechas
- [x] Barra de búsqueda con 4 criterios (nombre, género, precio, rating)
- [x] Tech Stack badges
- [x] Precios en MXN (×20)
- [x] Footer
- [x] `BASE_CSS` + CSS de categorías en `src/styles.py`

### Compañerx — ⏳ Login

- [ ] Validar credenciales
- [ ] `st.session_state["authenticated"] = True`
- [ ] Redirigir a homepage
- [ ] Usar `BASE_CSS` de `src/styles.py`

### Compañerx — ⏳ Análisis

- [ ] Cargar datos del juego desde `dataset_limpio.csv`
- [ ] Nube de palabras TF-IDF
- [ ] Gráfica de sentimiento VADER
- [ ] Tabla de reseñas
- [ ] Métricas del juego
- [ ] Descarga CSV/PNG
- [ ] Usar `BASE_CSS` de `src/styles.py`

## 7. Notas técnicas

### CSS compartido (`src/styles.py`)
- `BASE_CSS` contiene todo el diseño base: tipografía, colores, fondo, header, footer, cards, categorías (`.cat-row`, `.cat-track`, `.cat-arrow`), sidebar
- Cada página importa con `from src.styles import BASE_CSS`
- La homepage añade CSS específico (hero, featured, tech, scroll) concatenado

### Ejecución
```powershell
.venv\Scripts\activate
streamlit run app.py
```

### Convenciones de diseño
- No texto gris — todo `#e8eaf6` o brand colors
- Precios: USD × 20 → `MX$XXX`
- Fondo oscuro `#0b0e1a` con gradientes rosa/lila/cyan
- Fuentes: Rajdhani headings, Inter body, Orbitron tech accents